# S6E11 — exploring the data

Kaggle Playground Series S6E11, *Predicting Credit Card Fraud*.
Binary classification, scored on **ROC AUC**.

This notebook is the readable companion to `python -m scripts.profile_data`.
It computes the same quantities, so the two cannot disagree; the script writes
`eda/`, the notebook shows the reasoning.

Run it top to bottom. `data/train.csv` must already be present.

In [ ]:
# Notebooks live in notebooks/; the package lives at the repo root.
# Importing from src/ rather than re-implementing keeps the notebook and the
# scripts from drifting apart -- there is exactly one definition of the data
# handling, the models and the CV loop.
import sys, pathlib
ROOT = pathlib.Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from src import config
from src.data import load_raw, split_xy

pd.set_option("display.width", 120)
print("repo root:", ROOT)

## Load

In [ ]:
train, test = load_raw()
print(f"train {train.shape[0]:,} x {train.shape[1]}")
print(f"test  {test.shape[0]:,} x {test.shape[1]}")
train.head(3)

## The imbalance

This is the single most important property of the dataset and it shapes
everything downstream: the metric, the validation scheme, and which model
families are worth trying.

In [ ]:
y_all = train[config.TARGET_COL].astype(int)
pos, n = int(y_all.sum()), len(y_all)
rate = pos / n
print(f"positives      {pos:,} of {n:,}")
print(f"positive rate  {rate:.5f}   (1 in {1/rate:.1f})")
print(f"always-predict-majority accuracy = {1-rate:.5f}")
print()
print("That last number is why the competition is scored on ROC AUC and not")
print("accuracy: a model that never predicts fraud is already 98.3% accurate")
print("and completely useless.")

### A cross-check worth doing

`sample_submission.csv` ships a constant prediction. If that constant equals
the training positive rate, the organisers seeded it with the base rate — a
small confirmation that the train and test splits share a prevalence.

In [ ]:
sample = pd.read_csv(config.SAMPLE_SUBMISSION_CSV)
const = sample[config.TARGET_COL].iloc[0]
print(f"sample_submission constant : {const:.10f}")
print(f"train positive rate        : {rate:.10f}")
print(f"difference                 : {abs(const-rate):.2e}")
assert sample[config.TARGET_COL].nunique() == 1, "sample submission is not constant"

## Columns, types and missingness

In [ ]:
rows = []
for c in train.columns:
    if c in (config.ID_COL, config.TARGET_COL):
        continue
    s = train[c]
    rows.append({
        "column": c,
        "kind": "numeric" if pd.api.types.is_numeric_dtype(s) else "categorical",
        "unique": s.nunique(dropna=True),
        "missing": int(s.isna().sum()),
    })
schema = pd.DataFrame(rows).sort_values(["kind", "unique"])
print(f"total missing values across all columns: {schema['missing'].sum()}")
schema

## Univariate signal

Single-feature ROC AUC: how well each column alone separates fraud from
non-fraud. A value near 0.5 means no marginal signal; distance from 0.5 in
*either* direction is signal, since a reversed ranking is still a ranking.

**This is descriptive only.** Categorical columns are scored here by their
full-train target rate, which is leakage — fine for looking, not for modelling.

In [ ]:
from sklearn.metrics import roc_auc_score

scores = {}
for c in schema["column"]:
    s = train[c]
    if pd.api.types.is_numeric_dtype(s):
        v = s.fillna(s.median())
    else:
        v = s.map(train.groupby(s, observed=True)[config.TARGET_COL].mean())
    scores[c] = roc_auc_score(y_all, v)

uni = (pd.Series(scores, name="auc").to_frame()
         .assign(distance=lambda d: (d.auc - 0.5).abs())
         .sort_values("distance", ascending=False))
uni.head(12)

In [ ]:
top = uni.head(12).iloc[::-1]
fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top.index, top["distance"], color="#4C72B0")
ax.set_xlabel("|AUC - 0.5|   (univariate separation)")
ax.set_title("Strongest single features")
ax.grid(axis="x", alpha=0.3)
fig.tight_layout()

## Level effects

For low-cardinality columns, the fraud rate within each level, against the
1.67% base rate. Large lifts here are what make this problem tractable at all.

In [ ]:
LOW_CARD = [c for c in schema.query("unique <= 15")["column"]]
levels = []
for c in LOW_CARD:
    g = train.groupby(c, observed=True)[config.TARGET_COL].agg(["size", "mean"])
    g = g[g["size"] >= 20]
    for lvl, (sz, m) in g.iterrows():
        levels.append({"column": c, "level": lvl, "n": int(sz),
                       "fraud_rate": m, "lift": m / rate})
lv = pd.DataFrame(levels).sort_values("lift", ascending=False)
lv.head(15).assign(fraud_rate=lambda d: d.fraud_rate.round(4),
                   lift=lambda d: d.lift.round(1))

### Read the lift on the right scale

A 25x lift in *probability* space looks dramatic, but what a logistic
regression actually works in is **log-odds**. If the effect is roughly
constant there, a plain additive linear model can represent it fully, and
interaction terms will buy nothing.

This is the diagnostic that reshaped the S6E9 analysis; run it before
assuming feature crosses are needed.

In [ ]:
def logit(p):
    p = np.clip(p, 1e-9, 1-1e-9)
    return np.log(p / (1 - p))

col = "cvv_retry_count"   # the strongest lift in the table above
g = train.groupby(col, observed=True)[config.TARGET_COL].agg(["size", "mean"])
g = g[g["size"] >= 20]
g["lift_vs_base"] = g["mean"] / rate
g["logit"] = logit(g["mean"])
g["logit_shift"] = g["logit"] - logit(rate)
g.round({"mean": 4, "lift_vs_base": 1, "logit": 3, "logit_shift": 3})

## What this implies for modelling

Record the conclusions here before fitting anything, so the experiments in
`02_models.ipynb` are tests of stated predictions rather than a search for a
story afterwards. That ordering is what the project brief means by
*"conduct experiments to verify your motivations"*.